# ST Score Restore — P4 Clef Source Qualification

**Tek işlem:** Python 3.12 / CPU seçin ve `Run all` çalıştırın. Canonical ZIP veya Mac tarafından yeniden paketlenmiş ZIP, manifest ve 18 gerçek PNG teacher completion ile tam eşleşirse güvenle kabul edilir.


In [ ]:
# ST Score Restore — P4 clef source qualification (single Colab cell)
# Runtime: Python 3.12.x, CPU / Hardware accelerator: None

import hashlib
import json
import os
from pathlib import Path, PurePosixPath
import shutil
import subprocess
import sys
import tempfile
import zipfile


def fail(message: str) -> None:
    print(f"\nERROR CONTEXT: {message}")
    raise RuntimeError(message)


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_exact(path: Path, byte_size: int, sha256: str, label: str) -> None:
    if not path.is_file():
        fail(f"{label} bulunamadı: {path}")
    actual_size = path.stat().st_size
    actual_sha = sha256_file(path)
    if actual_size != byte_size or actual_sha != sha256:
        fail(
            f"{label} kimliği uyuşmuyor. "
            f"Beklenen size/SHA={byte_size}/{sha256}; "
            f"gerçek={actual_size}/{actual_sha}"
        )
    print(f"PASS — {label}: {path.name}")


def find_exact_artifact(
    preferred_path: Path,
    search_root: Path,
    byte_size: int,
    sha256: str,
    suffix: str,
    label: str,
) -> Path:
    """Find renamed/moved Drive artifacts without weakening byte identity."""
    if preferred_path.is_file():
        verify_exact(preferred_path, byte_size, sha256, label)
        return preferred_path
    matches = []
    if search_root.is_dir():
        for candidate in search_root.rglob(f"*{suffix}"):
            try:
                if candidate.is_file() and candidate.stat().st_size == byte_size:
                    if sha256_file(candidate) == sha256:
                        matches.append(candidate)
            except OSError:
                continue
    if not matches:
        fail(
            f"{label} bulunamadı. Beklenen SHA-256={sha256}. "
            f"Dosyayı {search_root} altına yükleyin."
        )
    selected = sorted(matches, key=lambda item: (len(item.parts), str(item)))[0]
    verify_exact(selected, byte_size, sha256, label)
    print(f"PASS — {label} SHA ile bulundu: {selected}")
    return selected


def _safe_real_zip_files(archive: zipfile.ZipFile) -> list[tuple[str, zipfile.ZipInfo]]:
    files = []
    for info in archive.infolist():
        name = info.filename
        pure = PurePosixPath(name)
        if name.startswith("/") or "\\" in name or ".." in pure.parts:
            raise RuntimeError(f"ZIP güvenlik ihlali: {name}")
        if info.is_dir() or name.startswith("__MACOSX/") or pure.name.startswith("._"):
            continue
        files.append((name, info))
    return files


def review_zip_payload_matches(path: Path, completion: dict) -> bool:
    """Accept a re-zipped transport only when manifest and all 18 PNG bytes match."""
    try:
        with zipfile.ZipFile(path, "r") as archive:
            files = _safe_real_zip_files(archive)
            manifests = [(name, info) for name, info in files if PurePosixPath(name).name == "review-bundle-manifest.v1.json"]
            if len(manifests) != 1:
                return False
            manifest = json.loads(archive.read(manifests[0][1]).decode("utf-8"))
            if manifest.get("schemaVersion") != "stage11.v2d.spatial-teacher-review-bundle.v1":
                return False
            if manifest.get("pageCount") != 18 or len(manifest.get("pages") or []) != 18:
                return False
            completion_pages = {str(page["pageId"]): page for page in completion.get("pages") or []}
            manifest_pages = {str(page["pageId"]): page for page in manifest.get("pages") or []}
            if len(completion_pages) != 18 or set(manifest_pages) != set(completion_pages):
                return False
            genuine_pngs = [(name, info) for name, info in files if name.lower().endswith(".png")]
            if len(genuine_pngs) != 18:
                return False
            for page_id, page in completion_pages.items():
                expected = page["reviewImage"]
                manifest_image = manifest_pages[page_id].get("reviewImage") or {}
                for key in ("path", "byteSize", "sha256", "width", "height"):
                    if manifest_image.get(key) != expected.get(key):
                        return False
                rel = str(expected["path"])
                matches = [(name, info) for name, info in genuine_pngs if name == rel or name.endswith("/" + rel)]
                if len(matches) != 1:
                    return False
                payload = archive.read(matches[0][1])
                if len(payload) != int(expected["byteSize"]):
                    return False
                if hashlib.sha256(payload).hexdigest() != str(expected["sha256"]):
                    return False
            return True
    except (OSError, KeyError, TypeError, ValueError, json.JSONDecodeError, zipfile.BadZipFile, RuntimeError):
        return False


def find_review_zip(preferred_path: Path, search_root: Path, completion: dict) -> Path:
    candidates = []
    if preferred_path.is_file():
        candidates.append(preferred_path)
    if search_root.is_dir():
        for candidate in search_root.rglob("*.zip"):
            if candidate.is_file() and candidate not in candidates:
                candidates.append(candidate)
    for candidate in sorted(candidates, key=lambda item: (len(item.parts), str(item))):
        if review_zip_payload_matches(candidate, completion):
            outer_sha = sha256_file(candidate)
            mode = "CANONICAL_OUTER_ZIP" if (
                candidate.stat().st_size == 18175904
                and outer_sha == "e5609d2f79139c209618a0fa8d61145ab55a6e548c0124e4664bb59d87366559"
            ) else "REPACKAGED_EXACT_PAYLOAD"
            print(f"PASS — teacher review ZIP: {candidate}")
            print(f"PASS — ZIP mode={mode}; outer SHA-256={outer_sha}")
            print("PASS — manifest + 18/18 gerçek PNG completion ile tam eşleşiyor")
            return candidate
    fail(
        "Teacher review ZIP bulunamadı veya içindeki manifest/18 PNG teacher completion ile eşleşmiyor. "
        f"ZIP dosyasını {search_root} altına koyun."
    )


def atomic_write(path: Path, payload: bytes) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, temp_name = tempfile.mkstemp(prefix=f".{path.name}.", suffix=".tmp", dir=path.parent)
    temp_path = Path(temp_name)
    try:
        with os.fdopen(fd, "wb") as handle:
            handle.write(payload)
            handle.flush()
            os.fsync(handle.fileno())
        os.replace(temp_path, path)
    finally:
        temp_path.unlink(missing_ok=True)


try:
    if sys.version_info[:2] != (3, 12):
        fail(
            f"Python {sys.version_info.major}.{sys.version_info.minor} desteklenmiyor. "
            "Colab: Runtime > Change runtime type > Runtime Version 2026.07 "
            "(Python 3.12), Hardware accelerator=None; sonra yeniden bağlanın."
        )
    print("PASS — runtime:", sys.version.split()[0], "CPU-only")

    subprocess.run(
        [sys.executable, "-m", "pip", "uninstall", "-y", "onnxruntime-gpu", "onnxruntime"],
        check=False,
        stdout=subprocess.DEVNULL,
    )
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "numpy>=2.0,<2.3",
            "onnxruntime==1.20.1",
            "opencv-python-headless==4.13.0.92",
            "scipy",
            "scikit-learn",
            "matplotlib",
            "pillow",
            "typing-extensions",
        ],
        check=True,
    )
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--no-deps",
            "git+https://github.com/BreezeWhite/oemer@dbe2a933d630d0f74805d717960eb259473f5978",
        ],
        check=True,
    )

    import cv2
    import onnxruntime as ort

    if ort.__version__ != "1.20.1":
        fail(f"onnxruntime sürümü yanlış: {ort.__version__}")
    providers = ort.get_available_providers()
    if "CPUExecutionProvider" not in providers or "CUDAExecutionProvider" in providers:
        fail(f"CPU-only ONNX Runtime gerekli; providers={providers}")
    from oemer.ete import generate_pred as _oemer_import_canary

    print("PASS — Oemer import ve CPU provider")

    from google.colab import drive

    if not Path("/content/drive/MyDrive").is_dir():
        try:
            drive.mount("/content/drive")
        except Exception as exc:
            fail(
                "Drive bağlanamadı. Runtime > Disconnect and delete runtime seçip "
                f"yeniden çalıştırın. Ayrıntı: {type(exc).__name__}: {exc}"
            )
    if not Path("/content/drive/MyDrive").is_dir():
        fail("Google Drive bağlı görünmüyor")
    print("PASS — Drive yalnız mounted filesystem olarak bağlı (ek Google API auth yok)")

    EVAL_ROOT = Path("/content/drive/MyDrive/ST_SCORE_RESTORE_STAGE11_EVAL")
    REVIEW_DIR = Path(
        "/content/drive/MyDrive/ST_SCORE_RESTORE_STAGE11_EVAL/"
        "V2D_TEACHER_SPATIAL_REVIEW"
    )
    COMPLETION = find_exact_artifact(
        REVIEW_DIR / "clef_box_teacher_completion.v1.json",
        EVAL_ROOT,
        70465,
        "68df771d5ace9f0b968452ff532fa2693fa2cd3405477d91fa3a98eccb190d54",
        ".json",
        "teacher completion",
    )
    completion_transport = json.loads(COMPLETION.read_text(encoding="utf-8"))
    REVIEW_ZIP = find_review_zip(
        REVIEW_DIR / "ST_SCORE_RESTORE_STAGE11_V2D_SPATIAL_TEACHER_REVIEW_PACKAGE_2026-09-09.zip",
        EVAL_ROOT,
        completion_transport,
    )
    OUTPUT_ROOT = Path(
        "/content/drive/MyDrive/ST_SCORE_RESTORE_STAGE11_EVAL/"
        "P4_CLEF_SOURCE_QUALIFICATION"
    )

    REPO = Path("/content/st-score-restore-engine-p4-d2da7f0")
    REPO_URL = "https://github.com/khfy7wpr5p-maker/st-score-restore-engine.git"
    PINNED_CODE_COMMIT = "d2da7f0a71170f61f10638dcb0cb85850d5ca5ca"
    if not REPO.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO)], check=True)
    if not (REPO / ".git").is_dir():
        fail(f"Repo yolu geçerli bir git deposu değil: {REPO}")
    subprocess.run(["git", "-C", str(REPO), "fetch", "-q", "origin", PINNED_CODE_COMMIT], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "-q", PINNED_CODE_COMMIT], check=True)
    head = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
    if head != PINNED_CODE_COMMIT:
        fail(f"Pinned repository HEAD uyuşmuyor: {head}")
    sys.path.insert(0, str(REPO / "src"))
    print("PASS — repo pinned HEAD:", head)

    from st_score_restore.stage11_v2d_clef_teacher_completion import (
        COMPLETION_IDENTITY,
        SOURCE_BUNDLE,
        validate_clef_teacher_completion,
        validate_clef_teacher_completion_binding,
        validate_clef_taxonomy_amendment,
    )
    from st_score_restore.stage11_v2d_clef_source_qualification import (
        EXPECTED_CLEF_BOX_COUNT,
        PRIMARY_IOU_THRESHOLD,
    )
    import st_score_restore.stage11_v2d_clef_source_qualification_colab as runner

    evidence_root = REPO / "evidence" / "stage11" / "v2d"
    amendment = json.loads(
        (evidence_root / "v2d-clef-teacher-taxonomy-amendment.v1.json").read_text(encoding="utf-8")
    )
    binding = json.loads(
        (evidence_root / "v2d-clef-box-teacher-completion-binding.v1.json").read_text(encoding="utf-8")
    )
    completion = completion_transport

    if validate_clef_taxonomy_amendment(amendment)["status"] != "pass":
        fail("taxonomy amendment doğrulanamadı")
    binding_result = validate_clef_teacher_completion_binding(binding)
    completion_result = validate_clef_teacher_completion(completion, amendment)
    if binding_result["status"] != "pass":
        fail("teacher binding doğrulanamadı")
    if completion_result["clefBoxCount"] != EXPECTED_CLEF_BOX_COUNT or EXPECTED_CLEF_BOX_COUNT != 213:
        fail("teacher clef box count 213 değil")
    if PRIMARY_IOU_THRESHOLD != 0.50:
        fail("Primary IoU sözleşmesi 0.50 değil")
    if completion.get("sourceReviewBundle") != SOURCE_BUNDLE:
        fail("completion içindeki source review bundle kimliği uyuşmuyor")
    print("PASS — teacher truth: 18 sayfa / 213 clef / IoU 0.50")

    runner.ROOT = OUTPUT_ROOT
    runner.INPUT_ROOT = OUTPUT_ROOT / "unused_source_assets"
    runner.SOURCE_ROOT = OUTPUT_ROOT / "exact_teacher_review_pages"
    runner.PROGRESS_ROOT = OUTPUT_ROOT / "page_results"
    runner.CHECKPOINT_ROOT = OUTPUT_ROOT / "oemer_checkpoints"
    runner.RESULT_PATH = OUTPUT_ROOT / "v2d_clef_source_qualification_result.json"
    runner.COMPLETION_PATH = COMPLETION

    expected_pages = {str(page["pageId"]): page for page in completion["pages"]}
    if len(expected_pages) != 18:
        fail("18 unique teacher page identity gerekli")

    with zipfile.ZipFile(REVIEW_ZIP, "r") as archive:
        files = _safe_real_zip_files(archive)

        staged = set()
        for page_id, page in expected_pages.items():
            image = page["reviewImage"]
            expected_rel = str(image["path"])
            matches = [(name, info) for name, info in files if name == expected_rel or name.endswith("/" + expected_rel)]
            if len(matches) != 1:
                fail(f"ZIP içinde exact tek PNG bulunamadı: {expected_rel}; eşleşme={len(matches)}")
            name, info = matches[0]
            if info.file_size != int(image["byteSize"]):
                fail(f"ZIP member boyutu uyuşmuyor: {page_id}")
            payload = archive.read(info)
            digest = hashlib.sha256(payload).hexdigest()
            if len(payload) != int(image["byteSize"]) or digest != str(image["sha256"]):
                fail(f"PNG SHA/size uyuşmuyor: {page_id}")
            decoded = cv2.imdecode(__import__("numpy").frombuffer(payload, dtype="uint8"), cv2.IMREAD_GRAYSCALE)
            if decoded is None:
                fail(f"PNG okunamıyor: {page_id}")
            height, width = decoded.shape
            if width != int(image["width"]) or height != int(image["height"]):
                fail(f"PNG dimensions uyuşmuyor: {page_id}")
            target = runner.SOURCE_ROOT / f"{page_id}.png"
            atomic_write(target, payload)
            staged.add(page_id)

    if staged != set(expected_pages):
        fail("18/18 exact PNG staging tamamlanmadı")
    print("PASS — 18/18 exact teacher-review PNG; PDF yeniden render edilmedi")

    # Canonical runnerın eski Drive-API indirme girişlerini kapatır. Girdiler yukarıda
    # exact ZIP byte'larından fail-closed doğrulanıp hazırlanmıştır.
    runner._drive_service = lambda: object()

    def mounted_filesystem_only(_service, _drive_id, path, byte_size, sha256):
        path = Path(path)
        if path == COMPLETION:
            verify_exact(path, int(byte_size), str(sha256), "teacher completion runtime input")
            return
        if path.parent == runner.INPUT_ROOT:
            if len(list(runner.SOURCE_ROOT.glob("*.png"))) != 18:
                fail("source asset bypass öncesi 18 exact PNG hazır değil")
            return
        fail(f"Beklenmeyen indirme isteği engellendi: {path}")

    runner._download_drive_file = mounted_filesystem_only

    result_path = runner.run()
    result = json.loads(result_path.read_text(encoding="utf-8"))
    if result["decisionBoundary"] != {
        "measurementComplete": True,
        "detectorQualified": False,
        "qualificationDecisionRequired": True,
        "restoredImagesEvaluated": False,
        "semanticPreservationEstablished": False,
        "overallStage11PassAuthorized": False,
        "productionReady": False,
        "stage12EntryAuthorized": False,
    }:
        fail("P4 decision boundary değişmiş")

    print("\n===== P4 TAMAMLANDI =====")
    print("POOLED:", result["pooledMetrics"])
    print("SAVED:", result_path)
    print("SHA-256:", sha256_file(result_path))
    print("detectorQualified: False — ayrı ölçüm kararı gerekiyor")

except Exception as exc:
    if not str(exc).startswith("ERROR CONTEXT"):
        print(f"\nERROR CONTEXT: {type(exc).__name__}: {exc}")
    raise
